In [41]:
import random
import joblib
from typing import Sequence

import matplotlib.pyplot as plt
import numpy
import pandas
from functools import lru_cache
from deap import algorithms
from deap import base, creator, tools
from scipy.stats import pointbiserialr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import roc_auc_score

# PREPARATION

In [42]:
# Define random seeds for reproducibility
SEED: int = 42
random.seed(SEED)
numpy.random.seed(SEED)

In [43]:
# Constants
#CSV_TRAIN_PATH: str = "diabetes/diabetes_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "diabetes/diabetes_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "Outcome"
#CONTINUOUS_COLS: list[str] = [
#    'Pregnancies',
#    'Glucose',
#    'BloodPressure',
#    'SkinThickness',
#    'Insulin',
#    'BMI',
#    'DiabetesPedigreeFunction',
#    'Age'
#]

CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
TARGET_COLUMN: str = "target_readmitted"
CONTINUOUS_COLS: list[str] = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'total_previous_visits'
]

# Bigger population -> bigger search space
POP_SIZE: int = 200
# Bigger generation number -> bigger convergence
NGEN: int = 100
# Crossing probability
CXPB: float = 0.5
# Mutation probability
MUTPB: float = 0.2

# Use ROC-AUC instead of PR-AUC
USE_ROC_AUC: bool = False

In [44]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

In [45]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [46]:
# Pre-compute folds and scaling
fold_indices = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)

for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    corr_series: pandas.Series = pandas.Series(dtype=float)
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col)
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

# MULTI OBJECTIVE

In [47]:
# Cache the results of evaluate_multi
def evaluate_multi(individual: Sequence[int]) -> tuple[float, float]:
    return _evaluate_multi_cached(tuple(individual))

@lru_cache(maxsize=None)
def _evaluate_multi_cached(individual: Sequence[int]) -> tuple[float, float]:
    if sum(individual) == 0:
        return 0.0, 0.0

    cols: numpy.ndarray = numpy.where(numpy.array(individual) == 1)[0]
    n_selected_features: int = len(cols)

    auc_scores: list[float] = []
    sign_scores: list[float] = []

    for fold_idx in range(len(fold_indices)):
        X_fold_train_scaled: numpy.ndarray = X_train_scaled_folds[fold_idx][:, cols]
        X_fold_val_scaled: numpy.ndarray = X_val_scaled_folds[fold_idx][:, cols]
        y_fold_train: numpy.ndarray = y_train_folds[fold_idx]
        y_fold_val: numpy.ndarray = y_val_folds[fold_idx]

        model: LogisticRegression = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            random_state=SEED)
        model.fit(X_fold_train_scaled, y_fold_train)

        probs: numpy.ndarray = model.predict_proba(X_fold_val_scaled)[:, 1]
        if USE_ROC_AUC:
            fold_auc: float = roc_auc_score(y_fold_val, probs)
        else:
            fold_auc: float = average_precision_score(y_fold_val, probs)
        auc_scores.append(fold_auc)

        # Get the correlation coefficients for the selected features in the current fold
        fold_corr: numpy.ndarray = corr_matrix[fold_idx, cols]

        # Multiply the correlation coefficients with the logistic regression coefficients
        check: numpy.ndarray = fold_corr * model.coef_[0]

        # Count penalties (where the product is negative or close to zero)
        penalties: int = numpy.sum((check < 0) | numpy.isclose(check, 0.0, atol=1e-12))

        # Calculate the score
        fold_sign: float = 1.0 - (penalties / n_selected_features)
        sign_scores.append(fold_sign)

    return numpy.mean(auc_scores), numpy.mean(sign_scores)

In [48]:
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))

if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox: base.Toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=X_search.shape[1],
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_multi)
toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", tools.mutFlipBit, indpb=1.0 / len(feature_names))
toolbox.register("select", tools.selNSGA2)

In [ ]:
pop: list[creator.Individual] = toolbox.population(n=POP_SIZE)

# Initial evaluation
invalid: list[creator.Individual] = [ind for ind in pop if not ind.fitness.valid]
fitnesses: list[tuple[float, float]] = list(map(toolbox.evaluate, invalid))
for ind, fit in zip(invalid, fitnesses):
    ind.fitness.values = fit

# Crowding distance assignment
pop: list[creator.Individual] = toolbox.select(pop, len(pop))  # crowding assignment

# Pareto archive
hof: tools.ParetoFront = tools.ParetoFront()
hof.update(pop)

# Genetic algorithm
for gen in range(NGEN):
    # Binary tournament selection (NSGA-II)
    offspring: list[creator.Individual] = tools.selTournamentDCD(pop, len(pop))
    offspring = [toolbox.clone(ind) for ind in offspring]

    # Crossover
    for ind1, ind2 in zip(offspring[::2], offspring[1::2]):
        if random.random() <= CXPB:
            toolbox.mate(ind1, ind2)
            del ind1.fitness.values
            del ind2.fitness.values

    # Mutation
    for mutant in offspring:
        if random.random() <= MUTPB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Fitness evaluation
    invalid: list[creator.Individual] = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses: list[tuple[float, float]] = list(map(toolbox.evaluate, invalid))
    for ind, fit in zip(invalid, fitnesses):
        ind.fitness.values = fit

    # Combine parents + offspring
    combined: list[creator.Individual] = pop + offspring

    # NSGA-II survival selection
    pop: list[creator.Individual] = toolbox.select(pop + offspring, POP_SIZE)

    # Update global Pareto archive
    hof.update(pop)

    print(f"Generation {gen + 1} done | Pareto size: {len(hof)}")

In [ ]:
pareto: list[creator.Individual] = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]

auc_values: list[float] = [ind.fitness.values[0] for ind in pareto]
sign_values: list[float] = [ind.fitness.values[1] for ind in pareto]

plt.scatter(auc_values, sign_values)
plt.xlabel("AUC-SCORE")
plt.ylabel("Coefficient Sign Consistency")
plt.title("Pareto Front")
plt.show()

In [ ]:
# Convert Pareto front fitness values to numpy array
points: numpy.ndarray = numpy.array([ind.fitness.values for ind in pareto])

# Extreme points
max_auc_index: int = int(numpy.argmax(points[:, 0]))
max_sign_index: int = int(numpy.argmax(points[:, 1]))

max_auc_ind: creator.Individual = pareto[max_auc_index]
max_sign_ind: creator.Individual = pareto[max_sign_index]

# Knee-point calculation
p1: numpy.ndarray = points[max_auc_index]
p2: numpy.ndarray = points[max_sign_index]

if numpy.allclose(p1, p2):
    knee_index: int = 0
else:
    line_vector: numpy.ndarray = p2 - p1
    line_vector = line_vector / numpy.linalg.norm(line_vector)

    distances: list[float] = []

    for p in points:
        vector: numpy.ndarray = p - p1
        projection: numpy.ndarray = numpy.dot(vector, line_vector) * line_vector
        orthogonal: numpy.ndarray = vector - projection
        distances.append(float(numpy.linalg.norm(orthogonal)))

    knee_index: int = int(numpy.argmax(distances))

knee_ind: creator.Individual = pareto[knee_index]

# Helper function to extract info
def extract_info(individual: creator.Individual) -> tuple[float, float, int, numpy.ndarray]:
    cols: numpy.ndarray = numpy.where(numpy.array(individual) == 1)[0]
    auc: float = individual.fitness.values[0]
    sign_consistency: float = individual.fitness.values[1]
    n_features: int = len(cols)
    return auc, sign_consistency, n_features, cols

auc1, sign1, nf1, cols1 = extract_info(max_auc_ind)
auc2, sign2, nf2, cols2 = extract_info(max_sign_ind)
auc3, sign3, nf3, cols3 = extract_info(knee_ind)

print("\n=== Pareto Front Model Comparison On Validation Set ===\n")

print("1) Maximum AUC model")
print(f"AUC on validation set: {auc1:.4f}")
print(f"Sign consistency on validation set: {sign1:.4f}")
print(f"Number of features: {nf1}")

print("\n2) Maximum Sign Consistency model")
print(f"AUC on validation set: {auc2:.4f}")
print(f"Sign consistency on validation set: {sign2:.4f}")
print(f"Number of features: {nf2}")

print("\n3) Knee-point model (balanced solution)")
print(f"AUC on validation set: {auc3:.4f}")
print(f"Sign consistency on validation set: {sign3:.4f}")
print(f"Number of features: {nf3}")

selected_feature_names: list[str] = [feature_names[i] for i in cols3]
selected_feature_names.sort()
print("Selected features for 'Knee-point model':", selected_feature_names)

In [ ]:
# Calculate the final results on the test set
def evaluate_on_test(cols: numpy.ndarray, should_save: bool = False) -> tuple[float, float]:
    X_search_reduced: numpy.ndarray = X_search[:, cols]
    X_test_reduced: numpy.ndarray = X_test.to_numpy()[:, cols]

    scaler: StandardScaler = StandardScaler()
    X_search_reduced_scaled: numpy.ndarray = scaler.fit_transform(X_search_reduced)
    X_test_reduced_scaled: numpy.ndarray = scaler.transform(X_test_reduced)

    model: LogisticRegression = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED)
    model.fit(X_search_reduced_scaled, y_search)

    test_probs: numpy.ndarray = model.predict_proba(X_test_reduced_scaled)[:, 1]
    roc_auc: float = roc_auc_score(y_test, test_probs)
    pr_auc: float = average_precision_score(y_test, test_probs)

    if should_save:
        joblib.dump({
            "model": model,
            "scaler": scaler,
            "features": [feature_names[i] for i in cols],
        }, "multi_objective_model_package.joblib")

    return roc_auc, pr_auc

roc_auc1_t, pr_auc1_t = evaluate_on_test(cols1)
roc_auc2_t, pr_auc2_t = evaluate_on_test(cols2)
roc_auc3_t, pr_auc3_t = evaluate_on_test(cols3, True)

print("\n=== Test Set Model Comparison ===\n")

print("1) Maximum AUC model")
print(f"ROC-AUC on test set: {roc_auc1_t:.4f}")
print(f"PR-AUC on test set: {pr_auc1_t:.4f}")
print(f"Number of features: {nf1}")

print("\n2) Maximum Sign Consistency model")
print(f"ROC-AUC on test set: {roc_auc2_t:.4f}")
print(f"PR-AUC on test set: {pr_auc2_t:.4f}")
print(f"Number of features: {nf2}")

print("\n3) Knee-point model (balanced solution)")
print(f"ROC-AUC on test set: {roc_auc3_t:.4f}")
print(f"PR-AUC on test set: {pr_auc3_t:.4f}")
print(f"Number of features: {nf3}")

# SINGLE OBJECTIVE

In [ ]:
# Reset random seeds for reproducibility
random.seed(SEED)
numpy.random.seed(SEED)

In [ ]:
# Cache the results of evaluate_multi
def evaluate_single(individual: Sequence[int]) -> tuple[float]:
    return _evaluate_single_cached(tuple(individual))

@lru_cache(maxsize=None)
def _evaluate_single_cached(individual: Sequence[int]) -> tuple[float]:
    if sum(individual) == 0:
        return (0.0,)

    cols: numpy.ndarray = numpy.where(numpy.array(individual) == 1)[0]

    auc_scores: list[float] = []

    for fold_idx in range(len(fold_indices)):
        X_fold_train_scaled: numpy.ndarray = X_train_scaled_folds[fold_idx][:, cols]
        X_fold_val_scaled: numpy.ndarray = X_val_scaled_folds[fold_idx][:, cols]
        y_fold_train: numpy.ndarray = y_train_folds[fold_idx]
        y_fold_val: numpy.ndarray = y_val_folds[fold_idx]

        model: LogisticRegression = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            random_state=SEED)
        model.fit(X_fold_train_scaled, y_fold_train)

        probs: numpy.ndarray = model.predict_proba(X_fold_val_scaled)[:, 1]

        if USE_ROC_AUC:
            fold_auc = roc_auc_score(y_fold_val, probs)
        else:
            fold_auc = average_precision_score(y_fold_val, probs)
        auc_scores.append(fold_auc)

    return (numpy.mean(auc_scores),)

In [ ]:
if "FitnessSingle" not in creator.__dict__:
    creator.create("FitnessSingle", base.Fitness, weights=(1.0,))

if "IndividualSingle" not in creator.__dict__:
    creator.create("IndividualSingle", list, fitness=creator.FitnessSingle)

In [ ]:
toolbox: base.Toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.IndividualSingle,
    toolbox.attr_bool,
    n=X_search.shape[1],
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_single)
toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", tools.mutFlipBit, indpb=1.0 / len(feature_names))
toolbox.register("select", tools.selTournament, tournsize=3)

pop: list[creator.IndividualSingle] = toolbox.population(n=POP_SIZE)
hof: tools.HallOfFame = tools.HallOfFame(1)

stats: tools.Statistics = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("max", numpy.max)

pop, logbook = algorithms.eaMuPlusLambda(
    pop,
    toolbox,
    mu=POP_SIZE,
    lambda_=POP_SIZE,
    cxpb=0.5,
    mutpb=0.2,
    ngen=NGEN,
    stats=stats,
    halloffame=hof,
    verbose=True
)

best_individual: creator.IndividualSingle = hof[0]
best_cols: list[int] = [i for i, bit in enumerate(best_individual) if bit == 1]

selected_feature_names: list[str] = [feature_names[i] for i in best_cols]
selected_feature_names.sort()
print("Selected features:", selected_feature_names)

In [ ]:
# Calculate the final results on the test set
X_search_reduced: numpy.ndarray = X_search[:, best_cols]
X_test_reduced: numpy.ndarray = X_test.to_numpy()[:, best_cols]

scaler: StandardScaler = StandardScaler()
X_search_reduced_scaled: numpy.ndarray = scaler.fit_transform(X_search_reduced)
X_test_reduced_scaled: numpy.ndarray = scaler.transform(X_test_reduced)

model: LogisticRegression = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    random_state=SEED)
model.fit(X_search_reduced_scaled, y_search)

test_probs: numpy.ndarray = model.predict_proba(X_test_reduced_scaled)[:, 1]
test_roc_auc: float = roc_auc_score(y_test, test_probs)
test_pr_auc: float = average_precision_score(y_test, test_probs)

print("\n=== Test Set Model Comparison ===\n")

print(f"ROC-AUC on test set: {test_roc_auc:.4f}")
print(f"PR-AUC on test set: {test_pr_auc:.4f}")
print(f"Number of features: {len(best_cols)}")

In [ ]:
# Saving best single objective model
joblib.dump({
    "model": model,
    "scaler": scaler,
    "features": [feature_names[i] for i in best_cols],
}, "single_objective_model_package.joblib")

In [ ]:
selected_feature_names_multi: list[str] = [feature_names[i] for i in cols3]
selected_feature_names_single: list[str] = [feature_names[i] for i in best_cols]

print("Selected features count 'Multi-objective model':", len(selected_feature_names_multi))
print("Selected features count 'Single-objective model':", len(selected_feature_names_single))
print("Continuous features count 'Multi-objective model':", len(set(CONTINUOUS_COLS) & set(selected_feature_names_multi)))

missing_from_single = list(set(selected_feature_names_multi) - set(selected_feature_names_single))
print("Missing from Multi:", missing_from_single)

missing_from_multi = list(set(selected_feature_names_single) - set(selected_feature_names_multi))
print("Missing from Single:", missing_from_multi)

multi_continuous_diff = list(set(CONTINUOUS_COLS) - set(selected_feature_names_multi))
print("Continuous features missing from Multi:", multi_continuous_diff)

single_continuous_diff = list(set(CONTINUOUS_COLS) - set(selected_feature_names_single))
print("Continuous features missing from Single:", single_continuous_diff)